# Causal ranking feature dataset

BT4222 Group 6 · feature engineering. Run the three `3_Representations` notebooks (3_01, 3_02, 3_03) first.

One shared feature API supports complete logged slates and unlabelled retrieved Top-M candidates.
Historical state always consumes the **complete earlier exposure stream**, even when emitted training
pairs are sampled. Validation/test slates stay complete. This notebook defaults to a chronological smoke
build; set `MAX_TIMESTAMPS=None` for all agreed data once the smoke audit passes.

The core method is implemented in the cells below. Shared imports handle file loading, cache/hash checks, storage, and audit reports. Build calls explicitly receive the notebook-defined functions. `REBUILD=False` reuses saved datasets; set it to `True` after editing a method and rebuild dependent outputs.

In [1]:
from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

# Find the source package whether this notebook starts in the root or its subfolder.
CODE_ROOT = next(
    (
        p
        for p in (
            Path.cwd(),
            *Path.cwd().parents,
            Path("/content/drive/MyDrive/BT4222"),
        )
        if (p / "src/newsrec").is_dir()
    ),
    None,
)
if CODE_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook from the project checkout containing src/newsrec/."
    )
sys.path.insert(0, str(CODE_ROOT / "src"))
from newsrec.common import Settings, saved_stage_ready

DATA_ROOT = (
    None  # Optional: point to another folder holding the project's saved datasets.
)
ROOT = (
    Path(DATA_ROOT or os.environ.get("NEWSREC_ROOT") or CODE_ROOT)
    .expanduser()
    .resolve()
)
ARTIFACTS = ROOT / "3_Representations/artifacts"
print("Data/output root:", ROOT)

from newsrec.features import feature_schema
from newsrec.io import feature_run, feature_sample
import hashlib
from collections import deque
from newsrec.common import DEFAULT_SETTINGS
from newsrec.temporal import TemporalState as _StateStorage, User, HOUR
from newsrec.features import prepare_feature_pairs
from newsrec.representations import Bundle
from newsrec.pipeline import replay_batches, time_horizon
from newsrec.common import normalise

Data/output root: /Users/bryan03/Desktop/BT4222-Group-6-


In [2]:
# Defaults live in src/newsrec/common.py; override with Settings(...) if needed.
SETTINGS = Settings()
REBUILD = False  # Reuse saved outputs; True regenerates this notebook's results.

## 1. Inspect the model contract

Core groups: content, history, match, repeated exposure, smoothed popularity and observed age.
Report extensions: three-interest matching and supported past-only half-life. Request context is provided
for ablation; slate relative advantage is opt-in (`context_features=true`) and uses the complete original
logged slate before sampling. At inference it must use the actual retrieved candidate set.

IDs, targets, shuffled position, slate length and current click-count summaries are outside the model allowlist.
Missing cosines/half-life use zero plus masks. Half-life compares disjoint completed six-hour windows,
uses Beta(1,1) smoothed CTR, and requires configurable support (100 exposures per window by default).
This specifies a new reproducible estimator; the report's original notebook is absent.

In [3]:
schema = feature_schema(SETTINGS)
display(pd.DataFrame(schema["features"]))
assert not set(schema["model_columns"]) & set(schema["excluded"])

,name,dtype,group,missing_value,definition,cutoff,preprocessing,consumer
0,category_code,float32,content,0,category code,static train-fitted content or events strictly...,none; fit any scaler on training rows only,ranker
1,subcategory_code,float32,content,0,subcategory code,static train-fitted content or events strictly...,none; fit any scaler on training rows only,ranker
2,title_word_count,float32,content,0,title word count,static train-fitted content or events strictly...,none; fit any scaler on training rows only,ranker
3,abstract_word_count,float32,content,0,abstract word count,static train-fitted content or events strictly...,none; fit any scaler on training rows only,ranker
4,has_abstract,float32,content,0,has abstract,static train-fitted content or events strictly...,none; fit any scaler on training rows only,ranker
5,n_annotated_entities,float32,content,0,n annotated entities,static train-fitted content or events strictly...,none; fit any scaler on training rows only,ranker
6,entity_coverage,float32,content,0,entity coverage,static train-fitted content or events strictly...,none; fit any scaler on training rows only,ranker
7,article_has_text,float32,content,0,article has text,static train-fitted content or events strictly...,none; fit any scaler on training rows only,ranker
8,article_has_entity,float32,content,0,article has entity,static train-fitted content or events strictly...,none; fit any scaler on training rows only,ranker
9,log_history_length,float32,history,0,log history length,static train-fitted content or events strictly...,log1p already applied,ranker


## 2. Replay and write features

Prequential mode freezes fitted representation/model parameters but updates observed feedback after each
timestamp. Frozen mode consumes earlier splits only and needs `TARGET_SPLIT`. Both modes advance the clock;
age and time-window queries still use the request time. The shared train/validation boundary second is processed
as one batch. `OVERWRITE=True` rebuilds only the selected run directory.

Training sampling is a seeded per-positive draw of up to four exposed negatives without replacement.
The pointwise feature table deduplicates the union of selected pairs; grouped models use the separate
positive-to-negative membership table. All popularity state is calculated before this sampling.

The SQLite user/article history bounds RAM; it requires local disk space. Full-run time, disk size and peak
RAM on MIND-large must be measured: a successful smoke run is not a full-scale performance claim.

### Causal profile state and past-only half-life

Repeated here so the ranking notebook can be reviewed on its own. The shared base only initialises storage. These overrides compute profiles, completed-hour half-life estimates, and timestamp-ordered feedback updates.

In [4]:
def normalised_mean(matrix, rows):
    if not len(rows):
        return (np.zeros(matrix.shape[1], dtype=np.float32), 0)
    selected = matrix[np.asarray(rows, dtype=np.int32)]
    valid = np.linalg.norm(selected, axis=1) > 1e-12
    if valid.any():
        pooled = normalise(selected[valid].mean(axis=0))
    else:
        pooled = np.zeros(matrix.shape[1], np.float32)
    return (pooled, int(valid.sum()))


class TemporalState(_StateStorage):
    """Visible causal calculations; shared base allocates arrays and the pair store."""

    def begin(self, requests, t):
        t = pd.Timestamp(t)
        if self.active_time is not None:
            raise ValueError("Previous timestamp has not been finished")
        if self.last_update is not None and t <= self.last_update:
            raise ValueError(
                "Replay timestamps must increase strictly; merge same-time requests"
            )
        if t < self.start or t > self.end:
            raise ValueError("Request outside state horizon")
        if (
            requests.impression_uid.duplicated().any()
            or not requests.timestamp.eq(t).all()
        ):
            raise ValueError("begin requires unique requests from this timestamp only")
        self.active_time = t
        for row in requests.itertuples(index=False):
            text = "" if pd.isna(row.history) else str(row.history)
            codes = tuple(int(token.removeprefix("N")) for token in text.split())
            key = int(row.user_code)
            if key in self.users:
                if self.users[key].snapshot_key != codes:
                    raise ValueError(f"Supplied history changed for user {key}")
                continue
            snapshot = self.bundle.rows(codes)
            user = User(snapshot, codes, deque(maxlen=self.cfg.history_cap))
            user.categories.update(self.cat[snapshot])
            user.subcategories.update(self.subcat[snapshot])
            self.users[key] = user

    def profile(self, user_code, t):
        t = pd.Timestamp(t)
        if self.active_time != t:
            raise ValueError(
                "Call begin() for this timestamp before requesting profiles"
            )
        key = int(user_code)
        user = self.users[key]
        cache_key = (key, user.version, t.value)
        if cache_key in self.profiles:
            return self.profiles[cache_key]
        observed = list(user.observed)
        seed = (
            user.snapshot
            if self.cfg.snapshot_order_verified
            else np.sort(user.snapshot)
        )
        history = np.r_[seed, [a for _, a in observed]].astype(np.int32)[
            -self.cfg.history_cap :
        ]
        recent = [
            a for time, a in observed if t.value - time <= self.cfg.recent_hours * HOUR
        ][-self.cfg.recent_count :]
        text, valid_text = normalised_mean(self.bundle.text, history)
        entity, valid_entity = normalised_mean(self.bundle.entity, history)
        long, _ = normalised_mean(self.bundle.text, user.snapshot)
        recent_vector, valid_recent = normalised_mean(self.bundle.text, recent)
        interests = []
        interest_categories = []
        for cat, _ in sorted(user.categories.items(), key=lambda x: (-x[1], x[0]))[
            : self.cfg.interest_count
        ]:
            rows = history[self.cat[history] == cat]
            vector, valid = normalised_mean(self.bundle.title, rows)
            if valid and np.linalg.norm(vector) > 0:
                interests.append(vector)
                interest_categories.append(cat)
        padded = np.zeros(self.cfg.history_cap, np.int32)
        padded[: len(history)] = history
        result = {
            "text": text,
            "entity": entity,
            "long_text": long,
            "recent_text": recent_vector,
            "interests": np.asarray(interests, np.float32).reshape(
                -1, self.cfg.text_dim
            ),
            "history_rows": padded,
            "history_mask": padded != 0,
            "selected_interest_categories": interest_categories,
            "recent_rows": list(map(int, recent)),
            "history_length": len(user.snapshot) + user.n_observed,
            "retained_length": len(history),
            "valid_text_fraction": valid_text / max(1, len(history)),
            "valid_entity_fraction": valid_entity / max(1, len(history)),
            "has_text": bool(np.linalg.norm(text) > 0),
            "has_entity": bool(np.linalg.norm(entity) > 0),
            "has_recent": bool(valid_recent and np.linalg.norm(recent_vector) > 0),
            "no_supplied_history": len(user.snapshot) == 0,
            "group_count": len(interests),
            "selected_group_count": min(self.cfg.interest_count, len(user.categories)),
            "state_version": user.version,
        }
        self.profiles[cache_key] = result
        return result

    def half_life(self, rows, t):
        t = pd.Timestamp(t)
        recent_end = int(t.value // HOUR - self.start_hour)
        recent_start = recent_end - 6
        result = []
        for row in rows:
            first = self.first[row]
            if first < 0:
                result.append((0.0, False, "no_prior_exposure", 0, 0))
                continue
            early_start = int((first + HOUR - 1) // HOUR - self.start_hour)
            early_end = early_start + 6
            if recent_start < early_end:
                result.append((0.0, False, "young_or_overlapping", 0, 0))
                continue
            early_e = int(self.hour_exposures[row, early_start:early_end].sum())
            recent_e = int(self.hour_exposures[row, recent_start:recent_end].sum())
            if min(early_e, recent_e) < self.cfg.half_life_min_exposures:
                result.append((0.0, False, "low_support", early_e, recent_e))
                continue
            early_c = int(self.hour_clicks[row, early_start:early_end].sum())
            recent_c = int(self.hour_clicks[row, recent_start:recent_end].sum())
            early = (early_c + 1) / (early_e + 2)
            recent = (recent_c + 1) / (recent_e + 2)
            if recent >= early:
                result.append((0.0, False, "no_decline", early_e, recent_e))
                continue
            estimate = np.log(2) * (recent_start - early_start) / np.log(early / recent)
            ok = bool(np.isfinite(estimate) and estimate > 0)
            result.append(
                (
                    float(estimate) if ok else 0.0,
                    ok,
                    "available" if ok else "invalid",
                    early_e,
                    recent_e,
                )
            )
        return result

    def finish(self, requests, events, t, feedback_splits):
        t = pd.Timestamp(t)
        if self.active_time != t:
            raise ValueError("Timestamp is not active")
        if not events.timestamp.eq(t).all() or not set(events.impression_uid) <= set(
            requests.impression_uid
        ):
            raise ValueError("Feedback must belong to this timestamp's requests")
        owners = requests.set_index("impression_uid").user_code
        if not np.array_equal(
            events.user_code.to_numpy(), events.impression_uid.map(owners).to_numpy()
        ):
            raise ValueError("Feedback/request user mismatch")
        if events.duplicated(["impression_uid", "news_code"]).any():
            raise ValueError("Duplicate feedback event")
        for u in requests.user_code.unique():
            self.users[int(u)].last_impression = t
        selected = events[events.split.isin(feedback_splits)]
        if len(selected):
            if not selected.clicked.isin([0, 1]).all():
                raise ValueError("Feedback labels must be binary")
            rows = self.bundle.rows(selected.news_code)
            labels = selected.clicked.to_numpy(np.int64)
            self.pairs.update(selected.user_code, rows, labels, t)
            np.add.at(self.exposures, rows, 1)
            np.add.at(self.clicks, rows, labels)
            unseen = self.first[rows] < 0
            self.first[rows[unseen]] = t.value
            hour = int(t.value // HOUR - self.start_hour)
            np.add.at(self.hour_exposures[:, hour], rows, 1)
            np.add.at(self.hour_clicks[:, hour], rows, labels)
            self.total_exposures += len(rows)
            self.total_clicks += int(labels.sum())
            clicked = selected.loc[selected.clicked.eq(1)].sort_values(
                ["user_code", "news_code", "impression_uid"]
            )
            for event in clicked.itertuples(index=False):
                user = self.users[int(event.user_code)]
                a = self.bundle.lookup[int(event.news_code)]
                user.observed.append((t.value, a))
                user.n_observed += 1
                user.categories[self.cat[a]] += 1
                user.subcategories[self.subcat[a]] += 1
                user.version += 1
        self.profiles.clear()
        self.last_update = t
        self.active_time = None

### Ranking-feature calculations

`prepare_feature_pairs` checks keys, timestamp alignment and catalogue eligibility. The formulas below use pre-feedback counts and profiles. `clicked` and slate position are never read by this function. Slate-relative advantage is calculated only when `context_features=True`.

In [5]:
def build_features(
    requests,
    candidate_pairs,
    state_before_t,
    representation_bundle,
    feature_settings=DEFAULT_SETTINGS,
    candidate_source="retrieved",
):
    """Same input contract for logged or unlabelled retrieved candidates.

    Caller begins one complete timestamp, builds all feature requests, then calls
    finish with logged feedback. This function neither reads labels nor mutates state.
    """
    state = state_before_t
    bundle = representation_bundle
    cfg = feature_settings
    pairs, rows, t = prepare_feature_pairs(
        requests, candidate_pairs, state, bundle, cfg, candidate_source
    )
    prior_pairs = state.pairs.lookup(pairs.user_code, rows)
    unique_rows, inverse = np.unique(rows, return_inverse=True)
    unique_half = state.half_life(unique_rows, t)
    half = [unique_half[i] for i in inverse]
    global_prior = (state.total_clicks + 1) / (state.total_exposures + 2)
    n = len(pairs)
    metadata = bundle.meta.loc[rows].reset_index(drop=True)
    quality = bundle.quality.loc[pairs.news_code.to_numpy()].reset_index(drop=True)
    article_e = state.exposures[rows]
    article_c = state.clicks[rows]
    first = state.first[rows]
    seen = first >= 0
    pair_e = np.asarray([x[0] for x in prior_pairs], np.int64)
    pair_c = np.asarray([x[1] for x in prior_pairs], np.int64)
    pair_last = np.asarray(
        [-1 if x[2] is None else x[2] for x in prior_pairs], np.int64
    )
    h = np.asarray([x[0] for x in half], np.float64)
    has_h = np.asarray([x[1] for x in half], bool)
    text_valid = np.linalg.norm(bundle.text[rows], axis=1) > 0
    entity_valid = np.linalg.norm(bundle.entity[rows], axis=1) > 0
    title_valid = np.linalg.norm(bundle.title[rows], axis=1) > 0
    ages = np.where(seen, (t.value - first) / HOUR, 0)
    values = {
        "impression_uid": pairs.impression_uid.to_numpy(np.int64),
        "news_code": pairs.news_code.to_numpy(np.int64),
        "user_code": pairs.user_code.to_numpy(np.int64),
        "article_row": rows,
        "timestamp": t,
        "candidate_source": candidate_source,
        "eligible": True,
        "category_code": metadata.category_code.to_numpy(),
        "subcategory_code": metadata.subcategory_code.to_numpy(),
        "title_word_count": metadata.title_word_count.to_numpy(),
        "abstract_word_count": metadata.abstract_word_count.to_numpy(),
        "has_abstract": metadata.has_abstract.to_numpy(),
        "n_annotated_entities": quality.n_annotated_entities.to_numpy(),
        "entity_coverage": quality.entity_coverage.to_numpy(),
        "article_has_text": text_valid,
        "article_has_entity": entity_valid,
        "prior_exposure_count": pair_e,
        "log_prior_exposure_count": np.log1p(pair_e),
        "log_prior_nonclick_count": np.log1p(pair_e - pair_c),
        "log_prior_pair_click_count": np.log1p(pair_c),
        "hours_since_last_exposure": np.where(
            pair_last >= 0, (t.value - pair_last) / HOUR, 0
        ),
        "no_prior_pair_exposure": pair_last < 0,
        "prior_article_exposures": article_e,
        "prior_article_clicks": article_c,
        "log_prior_article_exposures": np.log1p(article_e),
        "log_prior_article_clicks": np.log1p(article_c),
        "smoothed_prior_article_ctr": (article_c + cfg.ctr_strength * global_prior)
        / (article_e + cfg.ctr_strength),
        "article_low_support": article_e < 100,
        "population_startup": state.total_exposures == 0,
        "article_age_hours": ages,
        "no_prior_article_exposure": ~seen,
        "age_left_censored": first == state.start.value,
        "estimated_article_half_life_hours": h,
        "has_half_life": has_h,
        "stale_after_grace": has_h & (ages > h + 6),
        "half_life_reason": [x[2] for x in half],
        "half_life_early_support": [x[3] for x in half],
        "half_life_recent_support": [x[4] for x in half],
        "day_part": t.hour // 6,
        "weekday": t.weekday(),
    }
    user_columns = [
        "state_version",
        "history_length",
        "log_history_length",
        "retained_history_length",
        "valid_text_history_fraction",
        "valid_entity_history_fraction",
        "no_supplied_history",
        "no_causal_history",
        "no_text_profile",
        "no_entity_profile",
        "no_recent_profile",
        "category_affinity",
        "subcategory_affinity",
        "seen_category",
        "text_cosine",
        "entity_cosine",
        "long_term_cosine",
        "recent_cosine",
        "previously_clicked",
        "multi_interest_match_score",
        "interest_group_count",
        "selected_interest_group_count",
        "multi_interest_valid",
        "hours_since_previous_impression",
        "first_visit",
        "text_match_valid",
        "entity_match_valid",
        "long_term_match_valid",
        "recent_match_valid",
    ]
    values.update({name: np.zeros(n, np.float64) for name in user_columns})
    for user_code, positions in pairs.groupby("user_code", sort=False).indices.items():
        idx = np.asarray(positions)
        article_rows = rows[idx]
        user = state.users[int(user_code)]
        p = state.profile(user_code, t)
        total = p["history_length"]
        category_counts = np.asarray(
            [user.categories[c] for c in state.cat[article_rows]]
        )
        subcategory_counts = np.asarray(
            [user.subcategories[c] for c in state.subcat[article_rows]]
        )
        dynamic = {
            "state_version": p["state_version"],
            "history_length": total,
            "log_history_length": np.log1p(total),
            "retained_history_length": p["retained_length"],
            "valid_text_history_fraction": p["valid_text_fraction"],
            "valid_entity_history_fraction": p["valid_entity_fraction"],
            "no_supplied_history": p["no_supplied_history"],
            "no_causal_history": total == 0,
            "no_text_profile": not p["has_text"],
            "no_entity_profile": not p["has_entity"],
            "no_recent_profile": not p["has_recent"],
            "category_affinity": category_counts / max(1, total),
            "subcategory_affinity": subcategory_counts / max(1, total),
            "seen_category": category_counts > 0,
            "text_cosine": bundle.text[article_rows] @ p["text"],
            "entity_cosine": bundle.entity[article_rows] @ p["entity"],
            "long_term_cosine": bundle.text[article_rows] @ p["long_text"],
            "recent_cosine": bundle.text[article_rows] @ p["recent_text"],
            "text_match_valid": p["has_text"] & text_valid[idx],
            "entity_match_valid": p["has_entity"] & entity_valid[idx],
            "long_term_match_valid": bool(np.linalg.norm(p["long_text"]) > 0)
            & text_valid[idx],
            "recent_match_valid": p["has_recent"] & text_valid[idx],
            "previously_clicked": (pair_c[idx] > 0)
            | np.isin(article_rows, user.snapshot),
            "multi_interest_match_score": np.max(
                bundle.title[article_rows] @ p["interests"].T, axis=1
            )
            if p["group_count"]
            else np.zeros(len(idx)),
            "interest_group_count": p["group_count"],
            "selected_interest_group_count": p["selected_group_count"],
            "multi_interest_valid": bool(p["group_count"]) & title_valid[idx],
            "hours_since_previous_impression": (
                t - user.last_impression
            ).total_seconds()
            / 3600
            if user.last_impression is not None
            else 0,
            "first_visit": user.last_impression is None,
        }
        for name, value in dynamic.items():
            values[name][idx] = value
    result = pd.DataFrame(values)
    result["state_version"] = result.state_version.astype(np.int64)
    result["history_length"] = result.history_length.astype(np.int64)
    if cfg.context_features:
        grouped = result.groupby("impression_uid")
        valid = grouped.multi_interest_valid.transform("all")
        result["slate_context_valid"] = valid
        result["personalized_relative_advantage"] = (
            result.multi_interest_match_score
            - grouped.multi_interest_match_score.transform("median")
        ).where(valid, 0)
    columns = feature_schema(cfg)["model_columns"]
    result[columns] = result[columns].astype(np.float32)
    if not np.isfinite(result[columns].to_numpy()).all():
        raise ValueError("Nonfinite model features")
    return result

### Deterministic exposed-negative sampling

Training keeps positive pairs and up to four sampled exposed negatives per positive. Validation/test retain complete logged slates. Historical state still consumes all allowed logged feedback.

In [6]:
def sample_training_groups(candidates, cfg=DEFAULT_SETTINGS):
    """Reproducible per-positive exposed negatives, independent of input row order."""
    groups = []
    selected = set()
    for uid, slate in candidates.groupby("impression_uid", sort=True):
        if slate.news_code.duplicated().any() or not slate.clicked.isin([0, 1]).all():
            raise ValueError("Sampling requires unique binary-labelled logged pairs")
        positives = sorted(map(int, slate.loc[slate.clicked.eq(1), "news_code"]))
        negatives = np.sort(
            slate.loc[slate.clicked.eq(0), "news_code"].to_numpy(np.int64)
        )
        if not positives:
            raise ValueError("Logged training impression has no positives")
        for positive in positives:
            seed = int.from_bytes(
                hashlib.sha256(f"{cfg.seed}:{int(uid)}:{positive}".encode()).digest()[
                    :8
                ],
                "little",
            )
            rng = np.random.default_rng(seed)
            sampled = sorted(
                map(
                    int,
                    rng.choice(
                        negatives,
                        min(cfg.negative_count, len(negatives)),
                        replace=False,
                    ),
                )
            )
            groups.append(
                {
                    "impression_uid": int(uid),
                    "positive_news_code": positive,
                    "negative_news_codes": sampled,
                }
            )
            selected.add((int(uid), positive))
            selected.update((int(uid), n) for n in sampled)
    return (
        pd.DataFrame(
            groups,
            columns=["impression_uid", "positive_news_code", "negative_news_codes"],
        ),
        selected,
    )

In [7]:
# A notebook run defaults to a small smoke build; the terminal runner defaults to full.
MAX_TIMESTAMPS = (
    None
    if os.environ.get("NEWSREC_FULL_FEATURES") == "1"
    else int(os.environ.get("NEWSREC_SMOKE_TIMESTAMPS", "1000"))
)
TARGET_SPLIT = None  # Set validation/test with Settings(update_mode='frozen').
OVERWRITE = REBUILD or os.environ.get("NEWSREC_OVERWRITE_FEATURES") == "1"
run_dir = feature_run(
    ROOT,
    SETTINGS,
    target_split=TARGET_SPLIT,
    max_timestamps=MAX_TIMESTAMPS,
    rebuild=OVERWRITE,
    state_factory=TemporalState,
    feature_builder=build_features,
    negative_sampler=sample_training_groups,
)
print(run_dir)
display(json.loads((run_dir / "run_manifest.json").read_text())["rows_written"])

Loaded saved feature run: /Users/bryan03/Desktop/BT4222-Group-6-/4_Feature_Engineering/artifacts/prequential-all
/Users/bryan03/Desktop/BT4222-Group-6-/4_Feature_Engineering/artifacts/prequential-all


{'train': 13084988, 'validation': 13192987, 'test': 14085557}

## 3. Audit coverage, sampling and values

Reports describe all scored logged pairs before training sampling; target tables are stored separately.
Do not load the full MIND feature dataset into pandas at once: select columns or use Arrow/Polars scans.
A smoke starting at training start may contain no validation/test output; absent partitions are expected.

In [8]:
display(pd.read_csv(run_dir / "coverage.csv"))
display(pd.read_csv(run_dir / "feature_quality.csv").head(20))
display(pd.read_csv(run_dir / "feature_cost.csv"))
display(json.loads((run_dir / "sampling_summary.json").read_text()))
display(feature_sample(run_dir, split="train", rows=8))

,split,representation,cohort,denominator,valid,fallback,valid_fraction
0,test,entity,articles_unexposed_in_train,9819542,8269243,1550299,0.842121
1,test,entity,clicked_exposures,574845,495300,79545,0.861624
2,test,entity,new_users_vs_train,2306476,1820394,486082,0.789253
3,test,entity,no_causal_history,47385,37457,9928,0.790482
4,test,entity,returning_users_vs_train,11779081,9294705,2484376,0.789086
...,...,...,...,...,...,...,...
67,validation,text_profile,clicked_exposures,514658,513092,1566,0.996957
68,validation,text_profile,new_users_vs_train,2025182,1984857,40325,0.980088
69,validation,text_profile,no_causal_history,40325,0,40325,0.000000
70,validation,text_profile,returning_users_vs_train,11167805,11167805,0,1.000000


,split,feature,n,zero,min,max,zero_fraction,nonfinite_count
0,train,category_code,70314387,0,1.0,18.00000,0.000000,0
1,train,subcategory_code,70314387,0,2.0,283.00000,0.000000,0
2,train,title_word_count,70314387,0,2.0,37.00000,0.000000,0
3,train,abstract_word_count,70314387,3135169,0.0,401.00000,0.044588,0
4,train,has_abstract,70314387,3135169,0.0,1.00000,0.044588,0
5,train,n_annotated_entities,70314387,14822155,0.0,30.00000,0.210798,0
6,train,entity_coverage,70314387,15512921,0.0,1.00000,0.220622,0
7,train,article_has_text,70314387,0,1.0,1.00000,0.000000,0
8,train,article_has_entity,70314387,15512921,0.0,1.00000,0.220622,0
9,train,log_history_length,70314387,439694,0.0,6.84588,0.006253,0


,stage,wall_seconds,sampled_peak_process_rss_mb,platform,cpu,logical_cpus,ram_gb,run,artifact_bytes,emitted_rows,timestamp_batches
0,ranking_features,9787.377177,4266.852352,macOS-26.6.2-arm64-arm-64bit,arm,10,17.179869,prequential-all,2496290668,40363532,520464


{'all_negatives': 67445389,
 'kept_unique_negatives': 10215990,
 'positive_groups': 2868998,
 'negative_group_memberships': 10888738,
 'discarded_unique_negative_fraction': 0.8485294524730222}

,impression_uid,news_code,user_code,article_row,timestamp,candidate_source,eligible,category_code,subcategory_code,title_word_count,...,multi_interest_match_score,interest_group_count,selected_interest_group_count,multi_interest_valid,hours_since_previous_impression,first_visit,text_match_valid,entity_match_valid,long_term_match_valid,recent_match_valid
0,309963,117242,538931,93676,2019-11-13 09:16:46,logged,True,4.0,40.0,6.0,...,0.402017,3.0,3.0,1.0,98.736115,0.0,1.0,1.0,1.0,0.0
1,309963,127836,538931,102154,2019-11-13 09:16:46,logged,True,12.0,199.0,8.0,...,0.165245,3.0,3.0,1.0,98.736115,0.0,1.0,1.0,1.0,0.0
2,309963,43752,538931,35135,2019-11-13 09:16:46,logged,True,4.0,221.0,9.0,...,0.121406,3.0,3.0,1.0,98.736115,0.0,1.0,0.0,1.0,0.0
3,309963,42741,538931,34344,2019-11-13 09:16:46,logged,True,6.0,160.0,11.0,...,0.260349,3.0,3.0,1.0,98.736115,0.0,1.0,1.0,1.0,0.0
4,309963,110709,538931,88482,2019-11-13 09:16:46,logged,True,1.0,10.0,13.0,...,0.353956,3.0,3.0,1.0,98.736115,0.0,1.0,1.0,1.0,0.0
5,315949,10802,654623,8747,2019-11-13 09:16:46,logged,True,17.0,273.0,9.0,...,0.215941,3.0,3.0,1.0,14.183611,0.0,1.0,1.0,1.0,1.0
6,315949,46415,654623,37250,2019-11-13 09:16:46,logged,True,12.0,203.0,9.0,...,0.331407,3.0,3.0,1.0,14.183611,0.0,1.0,1.0,1.0,1.0
7,315949,53474,654623,42851,2019-11-13 09:16:46,logged,True,12.0,63.0,8.0,...,0.346249,3.0,3.0,1.0,14.183611,0.0,1.0,1.0,1.0,1.0


## 4. Verify unlabelled inference parity

The later retrieval notebook supplies `(impression_uid, news_code)` pairs from the eligible catalogue.
No label/position is required. This initial request example uses the same pairs as the logged slate to
verify identical features. For a different Top-M set, only optional candidate-set context may change.

In [9]:
# Execute the same visible functions on one complete timestamp, before feedback.
bundle = Bundle(ROOT, SETTINGS)
state = TemporalState(bundle, *time_horizon(ROOT), SETTINGS)
try:
    t, requests, logged = next(replay_batches(ROOT, SETTINGS))
    state.begin(requests, t)
    scored = build_features(requests, logged, state, bundle, SETTINGS, "logged")
    unlabelled = build_features(
        requests,
        logged[["impression_uid", "news_code"]],
        state,
        bundle,
        SETTINGS,
        "retrieved",
    )
    pd.testing.assert_frame_equal(
        scored[schema["model_columns"]], unlabelled[schema["model_columns"]]
    )
    # All requests at t have been scored before feedback at t is consumed.
    feedback_splits = {"train", "validation", "test"}
    if SETTINGS.update_mode == "frozen":
        feedback_splits.discard(TARGET_SPLIT)
    state.finish(requests, logged, t, feedback_splits)
    print("Logged/retrieved parity passed; feedback applied only after scoring.")
finally:
    state.close()

Logged/retrieved parity passed; feedback applied only after scoring.


## Handoff and limitations

Read `feature_schema.json` and `run_manifest.json` before loading any partitions. Use only `model_columns`
and apply train-fitted downstream preprocessing. Each run contains keyed labels, request profile references,
negative groups, sampling summary, coverage and quality audits. A run must have `run_status.json: complete`;
its manifest explicitly identifies smoke/full status and temporal mode. Unshown retrieved items remain unlabelled.
Graph embeddings, K-means, learned contextual encoders, ANN retrieval and model training belong to later work.
Core feature/runtime coverage does not establish ranking improvement or live CTR uplift.